# Validation cohort preparation and clustering

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Prepare cohort 3 trajectories, normalize them using discovery-cohort parameters, fit two k-means clusters in cohorts 1 and 2, and assign validation participants to the discovery centroids. Assess discovery-cluster stability with bootstrapping and a reduced-feature sensitivity analysis.

## Inputs

- `files/inputs/combined_all_cohorts.xlsx`: master cohort data.
- `files/inputs/Subscales_sample3.xlsx`: validation EPDS totals and subscales. The alternative discovery branch reads `Subskalen Hauptstudie.xlsx`.
- `files/interm_files/sub_means_stds_[1, 2].pkl`: discovery normalization parameters.
- `files/interm_files/subscales_full_trajectories_[1, 2].xlsx`: normalized discovery trajectories.

## Outputs

- `files/interm_files/full_subscales_trajectories_withoutnorm_[3].xlsx`: unnormalized validation features.
- `files/interm_files/full_trajectories_val[3].xlsx`: normalized validation features.
- `files/interm_files/kmeans[1, 2].xlsx` and `kmeans[3].xlsx`: discovery and validation cluster assignments (`IDX`, coded 0/1).
- `files/plots/kmeans_[3].png`: validation cluster counts by diagnosis.
- `files/plots/Supplementary_Figure_Correlation_Matrix.png`: discovery feature correlations.

Other plots and bootstrap summaries are displayed in the notebook. 


## 1. Setup

In [ ]:
import pandas as pd

import trajectory_clustering_utils as hc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.cluster.vq import kmeans, vq
from sklearn.metrics import silhouette_samples
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly as py
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from sklearn.metrics import cohen_kappa_score


In [ ]:
from pathlib import Path

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_FILES.mkdir(parents=True, exist_ok=True)
OUTPUT_PLOTS.mkdir(parents=True, exist_ok=True)


## 2. Select and prepare cohort 3

Read the master workbook and harmonize the first 23 EPDS workbook columns. The first data row is removed according to the supplied workbook layout. Convert scores to numeric and retain complete subscale records.

In [ ]:
combined_cohorts = pd.read_excel(INPUT_PATH / 'combined_all_cohorts.xlsx')
coh = [3]
df = combined_cohorts[combined_cohorts['Cohort'].isin(coh)]
df


In [ ]:
if coh == [3]:
    subscales = pd.read_excel(INPUT_PATH / 'Subscales_sample3.xlsx')
else:
    subscales = pd.read_excel(INPUT_PATH / 'Subskalen Hauptstudie.xlsx')

subscales.columns
subscales = subscales.iloc[:, :23]

if coh==[3]:
    rename_dict = {
        'EPDS T0 - Anamnesebogen! -': 'EPDStotal_t0',
        'Unnamed: 4': 'Anhedonie_t0',
        'Unnamed: 5': 'Anxiety_t0',
        'Unnamed: 6': 'Depression_t0',
        'EPDS T1': 'EPDStotal_t1',
        'Unnamed: 8': 'Anhedonie_t1',
        'Unnamed: 9': 'Anxiety_t1',
        'Unnamed: 10': 'Depression_t1',
        'EPDS T2': 'EPDStotal_t2',
        'Unnamed: 12': 'Anhedonie_t2',
        'Unnamed: 13': 'Anxiety_t2',
        'Unnamed: 14': 'Depression_t2',
        'EPDS T3': 'EPDStotal_t3',
        'Unnamed: 16': 'Anhedonie_t3',
        'Unnamed: 17': 'Anxiety_t3',
        'Unnamed: 18': 'Depression_t3',
        'EPDS T4': 'EPDStotal_t4',
        'Unnamed: 20': 'Anhedonie_t4',
        'Unnamed: 21': 'Anxiety_t4',
        'Unnamed: 22': 'Depression_t4'
    }
    subscales.rename(columns=rename_dict, inplace=True)

    subscales = subscales.drop(index = 0)

    subscales = subscales.reset_index(drop=True)
else:
        rename_dict = {
            'EPDS T0': 'EPDStotal_t0',
            'Unnamed: 4': 'Anhedonie_t0',
            'Unnamed: 5': 'Anxiety_t0',
            'Unnamed: 6': 'Depression_t0',
            'EPDS T1': 'EPDStotal_t1',
            'Unnamed: 8': 'Anhedonie_t1',
            'Unnamed: 9': 'Anxiety_t1',
            'Unnamed: 10': 'Depression_t1',
            'EPDS T2': 'EPDStotal_t2',
            'Unnamed: 12': 'Anhedonie_t2',
            'Unnamed: 13': 'Anxiety_t2',
            'Unnamed: 14': 'Depression_t2',
            'EPDS T3': 'EPDStotal_t3',
            'Unnamed: 16': 'Anhedonie_t3',
            'Unnamed: 17': 'Anxiety_t3',
            'Unnamed: 18': 'Depression_t3',
            'EPDS T4': 'EPDStotal_t4',
            'Unnamed: 20': 'Anhedonie_t4',
            'Unnamed: 21': 'Anxiety_t4',
            'Unnamed: 22': 'Depression_t4'
        }
        subscales.rename(columns=rename_dict, inplace=True)

        subscales = subscales.drop(index = 0)

        subscales = subscales.reset_index(drop=True)
subscales


In [ ]:
if coh == [3]:
    subscales_EPDS = subscales.iloc[:, 3:]
    subscales_EPDS = subscales_EPDS.apply(pd.to_numeric, errors='coerce')
    subscales_EPDS['ID'] = subscales['BH-Code']
    subscales_EPDS = subscales_EPDS.dropna()
    subscales_EPDS
else:
    df_dropped = subscales.drop(columns=['VP-Nummer'])

    test = df_dropped[['EPDStotal_t0', 'Anhedonie_t0',
       'Anxiety_t0', 'Depression_t0', 'EPDStotal_t1', 'Anhedonie_t1',
       'Anxiety_t1', 'Depression_t1', 'EPDStotal_t2', 'Anhedonie_t2',
       'Anxiety_t2', 'Depression_t2', 'EPDStotal_t3', 'Anhedonie_t3',
       'Anxiety_t3', 'Depression_t3', 'EPDStotal_t4', 'Anhedonie_t4',
       'Anxiety_t4', 'Depression_t4']].apply(pd.to_numeric, errors='coerce')

    print("Data types after conversion:")
    print(test.dtypes)
    test["ID"] = df_dropped["ID"]

    subscales_EPDS = test.dropna()

subscales_EPDS.columns


### Eligibility and daily measurements

Require Mood Bool, Stress Bool, and Dem Bool to equal True, with nonmissing ID and CODE. Select mood and stress measurements from days 1–88.

In [ ]:
df_cohort = hc.assert_variables(df)

interval_size = 21
start_day = 1
finish_day = 88
df_cohort


## 3. Construct baseline and follow-up trajectories

For each participant and attribute, the first observed numeric score in day order is T0. Mask that score and preceding days before calculating follow-up means. Intervals remain anchored to study days, not the individual baseline date.

In [ ]:
def separate_baseline_from_traj(dataframe, choose_attribute):

    # Preserve day order when identifying each participant's baseline.
    columns = [col for col in dataframe.columns if choose_attribute in col]
    scores = dataframe[columns].apply(pd.to_numeric, errors='coerce')

    first_non_nan_values = []
    first_non_nan_indices = []

    for _, row in scores.iterrows():
        non_nan = row.dropna()
        if not non_nan.empty:
            first_non_nan_values.append(non_nan.iloc[0])
            first_non_nan_indices.append(non_nan.index[0])
        else:
            first_non_nan_values.append(np.nan)
            first_non_nan_indices.append(None)

    df_baseline = pd.DataFrame({
        f'{choose_attribute} T0': first_non_nan_values,
        'CODE': dataframe['CODE'],
        'ID': dataframe['ID']
    })

    traj_rows = []
    for idx, start_col in enumerate(first_non_nan_indices):
        row_values = scores.iloc[idx].copy()
        if start_col is not None:
            start_pos = columns.index(start_col) + 1
            # Exclude baseline and earlier days from follow-up means.
            row_values.iloc[:start_pos] = np.nan
        traj_rows.append(row_values)

    df_trajectory = pd.DataFrame(traj_rows, columns=columns)
    df_trajectory['CODE'] = dataframe['CODE']
    df_trajectory['ID'] = dataframe['ID']

    return df_baseline, df_trajectory


In [ ]:
df_cohort = hc.assert_variables(df)


Average available follow-up observations over days 1–21, 22–42, 43–63, and 64–84. The incomplete final interval is discarded. One observation can supply an interval mean; retain complete rows across the interval columns produced by the helper.

In [ ]:
interval_size = 21
start_day = 1
finish_day = 88
df_cohort

stress = hc.select_mood_stress_values('Stress', df_cohort, interval_size, start_day, finish_day)
df_baseline_stress, df_trajectory_stress = separate_baseline_from_traj(stress, 'Stress')

mood = hc.select_mood_stress_values('Mood', df_cohort, interval_size, start_day, finish_day)
df_baseline_mood, df_trajectory_mood = separate_baseline_from_traj(mood, 'Mood')

long_df_stress, wide_df_stress = hc.clean_mood_stress('Stress', df_trajectory_stress, interval_size, start_day, finish_day)
long_df_mood, wide_df_mood = hc.clean_mood_stress("Mood",  df_trajectory_mood, interval_size, start_day, finish_day)
wide_df_stress


Rename the zero-based intervals to T1–T4 and rejoin T0 on ID/CODE. Run renaming cells only once per preparation pass.

In [ ]:
old_columns = [col for col in wide_df_stress.columns if col.startswith('Stress T')]
print(old_columns)

new_columns = [f'Stress T{int(col.split("T")[1]) + 1}' for col in old_columns]

rename_dict = dict(zip(old_columns, new_columns))
wide_df_stress = wide_df_stress.rename(columns=rename_dict)

wide_df_stress


In [ ]:
old_columns = [col for col in wide_df_mood.columns if col.startswith('Mood T')]
print(old_columns)

new_columns = [f'Mood T{int(col.split("T")[1]) + 1}' for col in old_columns]

rename_dict = dict(zip(old_columns, new_columns))
wide_df_mood = wide_df_mood.rename(columns=rename_dict)
wide_df_mood


In [ ]:
df_baseline_stress = df_baseline_stress.dropna()
df_baseline_mood = df_baseline_mood.dropna()
wide_df_stress = wide_df_stress.dropna()
wide_df_mood = wide_df_mood.dropna()

merged_df_stress = pd.merge(df_baseline_stress, wide_df_stress, on=['CODE', 'ID'], how='inner')
merged_df_mood = pd.merge(df_baseline_mood, wide_df_mood, on=['CODE', 'ID'], how='inner')

merged_df_mood


## 4. Prepare questionnaire scores and align participants

Retain complete MPAS T1–T4 and master-workbook EPDS T0–T4 records. The master EPDS merge is retained as an intermediate table; the exported feature table uses the separate subscale workbook. Intersect IDs across mood, stress, MPAS, and EPDS subscales, then merge on ID/CODE. IDs are assumed unique.

In [ ]:
MPAS_parameters = ['MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4']

mpas = df_cohort[MPAS_parameters]
mpas = mpas.apply(pd.to_numeric, errors='coerce')
mpas['ID'] = df_cohort['ID']
mpas['CODE'] = df_cohort['CODE']
mpas = mpas.dropna()

MPAS_parameters = [ 'ID', "CODE", 'MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4']
wide_df_mpas = hc.mpas_epds_data_for_clustering(mpas, MPAS_parameters)

wide_df_mpas


In [ ]:
EPDS_parameters = ['EPDS T0', 'EPDS T1', 'EPDS T2', 'EPDS T3', 'EPDS T4']

epds = df_cohort[EPDS_parameters]
epds = epds.apply(pd.to_numeric, errors='coerce')
epds['ID'] = df_cohort['ID']
epds['CODE'] = df_cohort['CODE']
epds = epds.dropna()

EPDS_parameters = [ 'ID', "CODE",'EPDS T0','EPDS T1', 'EPDS T2', 'EPDS T3', 'EPDS T4']
wide_df_epds = hc.mpas_epds_data_for_clustering(epds, EPDS_parameters)

wide_df_epds


In [ ]:
df = [merged_df_mood,merged_df_stress, wide_df_epds, wide_df_mpas]
merged = pd.merge(df[0], df[1], on=['ID', "CODE"], how='inner')

for i in range(2, len(df)):
     merged = pd.merge(merged, df[i], on=['ID', "CODE"], how='inner')


In [ ]:
dfs = [merged_df_mood,merged_df_stress, wide_df_mpas, subscales_EPDS]
# Retain IDs available across all required feature sources.
common_ids = set.intersection(set(wide_df_mood['ID']),set(wide_df_mpas["ID"]), set(wide_df_stress["ID"]), set(subscales_EPDS["ID"]))

epds_sub_fold = subscales_EPDS[subscales_EPDS["ID"].isin(common_ids)]
stress_f = merged_df_stress[merged_df_stress['ID'].isin(common_ids)]
mood_f = merged_df_mood[merged_df_mood['ID'].isin(common_ids)]
mpas_f = wide_df_mpas[wide_df_mpas['ID'].isin(common_ids)]

epds_sub_f = pd.merge(epds_sub_fold, stress_f[['ID', 'CODE']], on='ID', how='inner')


In [ ]:
df_new = [epds_sub_f,stress_f, mood_f,mpas_f ]
merged_new = pd.merge(df_new[0], df_new[1], on=['ID', "CODE"], how='inner')

for i in range(2, len(df_new)):
     merged_new = pd.merge(merged_new, df_new[i], on=['ID', "CODE"], how='inner')


In [ ]:
merged_new

output_path = OUTPUT_FILES / f'full_subscales_trajectories_withoutnorm_[3].xlsx'
merged_new.to_excel(output_path)
merged_new


## 5. Normalize using discovery parameters

Apply `(value − discovery mean) / discovery SD` separately to each feature block; do not estimate validation-specific moments. The helper standardizes numeric columns and restores ID/CODE. Outer joins retain records present in any normalized block, so unmatched records can introduce missing features.

In [ ]:
output_pkl = OUTPUT_FILES / f'sub_means_stds_[1, 2].pkl'

with open(output_pkl, 'rb') as f:
    means_stds = pickle.load(f)
means_stds


In [ ]:
anhedonia_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Anhedonie")]
anhedonia_sub["ID"] = epds_sub_f["ID"]
anhedonia_sub["CODE"] = epds_sub_f["CODE"]

anxiety_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Anxiety")]
anxiety_sub["ID"] = epds_sub_f["ID"]
anxiety_sub["CODE"] = epds_sub_f["CODE"]

depression_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Depression")]
depression_sub["ID"] = epds_sub_f["ID"]
depression_sub["CODE"] = epds_sub_f["CODE"]

# Reuse discovery scaling parameters for the validation subscales.
norm_anhedonia = hc.normalization_validation(anhedonia_sub, means_stds['mean_anh'], means_stds['std_anh'])
norm_anxiety = hc.normalization_validation(anxiety_sub, means_stds['mean_anx'], means_stds['std_anx'])
norm_depression = hc.normalization_validation(depression_sub, means_stds['mean_dep'], means_stds['std_dep'])
means_stds['mean_anh']


In [ ]:
norm_mpas = hc.normalization_validation(mpas_f, means_stds['mean_mpas'], means_stds['std_mpas'])

stress_columns = [col for col in stress_f.columns if 'Stress' in col]
stress_f[stress_columns] = stress_f[stress_columns].apply(pd.to_numeric, errors='coerce')
norm_stress = hc.normalization_validation(stress_f, means_stds['mean_stress'], means_stds['std_stress'])

mood_columns = [col for col in mood_f.columns if 'Mood' in col]
mood_f[mood_columns] = mood_f[mood_columns].apply(pd.to_numeric, errors='coerce')
norm_mood = hc.normalization_validation(mood_f, means_stds['mean_mood'], means_stds['std_mood'])

norm_mood


In [ ]:
norm_stress


In [ ]:
dfs = [norm_anhedonia, norm_anxiety, norm_depression, norm_mpas, norm_mood, norm_stress]
merged_df = pd.merge(dfs[0], dfs[1], on=['ID', "CODE"], how='outer')

for i in range(2, len(dfs)):
    merged_df = pd.merge(merged_df, dfs[i], on=['ID', "CODE"], how='outer')

output_path = OUTPUT_FILES / f'full_trajectories_val{coh}.xlsx'
merged_df.to_excel(output_path)
merged_df


## 6. Fit discovery k-means

Use **k=2**, carried forward from the discovery silhouette analysis. Select 25 features: anhedonia, depression, anxiety, mood, and stress at five time points each. MPAS is prepared and exported but excluded from clustering here. Convert to float32 and fit SciPy k-means with NumPy seed 42. Infinite values become NaN; no imputation or subsequent missing-row removal is active, so clustering requires finite inputs.

In [ ]:
data_path_trajectories = OUTPUT_FILES / 'subscales_full_trajectories_[1, 2].xlsx'
full_df_trajectories1 = pd.read_excel(data_path_trajectories)
df_first = full_df_trajectories1.select_dtypes(include=['number'])
col_names = full_df_trajectories1.columns.tolist()
df_first

anhedonia_columns = [col for col in col_names if 'Anhedonie' in col]
depression_columns = [col for col in col_names if 'Depression' in col]
anxiety_columns = [col for col in col_names if 'Anxiety' in col]
stress_columns = [col for col in col_names if 'Stress' in col]
mood_columns = [col for col in col_names if 'Mood' in col]

# Preserve this feature order for validation assignment.
columns =  anhedonia_columns + depression_columns+ anxiety_columns + mood_columns + stress_columns

custom_palette = {'PPD': '#E2725B',
                  'ND': '#4682B4',
                  'AS': '#6B8E23'}

selected_df = df_first[columns]

selected_df.replace([np.inf, -np.inf], np.nan, inplace=True)

selected_df.columns

data_tra = np.array(selected_df.values, dtype = np.float32)

selected_df


In [ ]:
np.size(data_tra, 1)


Fit discovery centroids and record assignments. Diagnosis-plot bars show counts; annotations show each cluster’s percentage within a diagnosis, despite the percentage axis label.

In [ ]:
np.random.seed(42)
centroids_first, _ = kmeans(data_tra, 2)
IDX, _ = vq(data_tra, centroids_first)

idx_counts = pd.Series(IDX).value_counts()
print(idx_counts)
selected_df["IDX"] = IDX
selected_df["CODE"] = full_df_trajectories1["CODE"]
selected_df['ID'] = full_df_trajectories1["ID"]
selected_df["IDX"] = IDX
print(selected_df["IDX"].value_counts()
)
selected_df.columns

custom_palette = {
    'PPD': '#FF7F50',
    'ND':  '#0056A0',
    'AS':  '#2C9F6D'
}

hue_order = ['ND', 'AS', 'PPD']

diagnosis_totals = selected_df['CODE'].value_counts().to_dict()

ax = sns.countplot(data=selected_df, x='IDX', hue='CODE',
                   palette=custom_palette, hue_order=hue_order)

ax.set_xticklabels(['Cluster 1', 'Cluster 2'])

for i, container in enumerate(ax.containers):
    diag = hue_order[i]
    total = diagnosis_totals.get(diag, 1)
    for bar in container:
        height = bar.get_height()
        percentage = (height / total) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 1,
            f'{percentage:.1f}%',
            ha='center', va='bottom',
            fontsize=12
        )

ax.set_xlabel("")

ax.set_ylabel("Percentage within diagnosis", fontsize=13)
ax.set_yticks([])

sns.despine()

ax.legend(title='Diagnosis', fontsize=11, title_fontsize=12)

plt.title('Diagnosis distribution per Cluster - Discovery Cohort', fontsize=16)

plt.tight_layout()
plt.show()


## 7. Bootstrap discovery-cluster stability

Refit k-means in 10,000 participant resamples with replacement and compare labels against the corresponding original labels using adjusted Rand index (ARI). **This cell assumes exactly 420 discovery rows**; the fixed sample size is preserved. The printed “95% CI” is the 2.5th–97.5th percentile range of bootstrap ARIs.

In [ ]:
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)

centroids, _ = kmeans(data_tra, 2)
original_labels, _ = vq(data_tra, centroids)

ari = []

for i in range(10000):

    # Fixed discovery sample size in the original analysis.
    idx = np.random.choice(420, 420)
    boot_data = data_tra[idx]

    boot_centroids, _ = kmeans(boot_data, 2)
    boot_labels, _ = vq(boot_data, boot_centroids)

    # Compare labels on the same resampled participants.
    ari.append(adjusted_rand_score(original_labels[idx], boot_labels))

print("Mean ARI:", np.mean(ari))
print("Median ARI:", np.median(ari))
print("95% CI:", np.percentile(ari, [2.5, 97.5]))


In [ ]:
plt.hist(ari, bins=30)
plt.axvline(np.median(ari), linestyle="--", label="Median ARI")
plt.xlabel("Adjusted Rand Index (ARI)")
plt.ylabel("Frequency")
plt.title("Bootstrap Stability of K-means Clustering")
plt.legend()
plt.show()


In [ ]:
data_path_save  = OUTPUT_FILES / f'kmeans[1, 2].xlsx'
selected_df.to_excel(data_path_save, index=False)
selected_df['CODE'].value_counts()


## 8. Assign validation participants

Select validation features in exactly the discovery column order, then assign each participant to the nearest discovery centroid using `vq`. No validation k-means model is fitted. Save assignments and the diagnosis count plot.

In [ ]:
df_sec = merged_df.select_dtypes(include=['number'])
selected_df_sec = df_sec[columns]

selected_df_sec.replace([np.inf, -np.inf], np.nan, inplace=True)

selected_df_sec.columns
data_tra_sec = np.array(selected_df_sec.values, dtype = np.float32)
selected_df_sec.columns


In [ ]:
data_tra_sec = np.array(selected_df_sec.values, dtype = np.float32)
# Assign to the discovery centroids without refitting.
IDX_sec, _ = vq(data_tra_sec, centroids_first)

idx_counts = pd.Series(IDX_sec).value_counts()
print('c', idx_counts)

selected_df_sec["IDX"] = IDX_sec
selected_df_sec["CODE"] = merged_df["CODE"]
selected_df_sec['ID'] = merged_df["ID"]

print(selected_df_sec["IDX"].value_counts()
)

plt.figure(figsize=(10, 6))
ax = sns.countplot(data=selected_df_sec, x="IDX", hue="CODE", palette=custom_palette, orient='h')

for container in ax.containers:
    ax.bar_label(container, padding=2, fontsize=14, color='black')

plt.ylabel("Number of Participants")
plt.xlabel("Cluster")
plt.title("Clusters of Participants from Validation Cohort")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.savefig(OUTPUT_PLOTS / f'kmeans_{coh}.png', dpi=300)

plt.show()

selected_df_sec


In [ ]:
data_path_save  = OUTPUT_FILES / f'kmeans[3].xlsx'
selected_df_sec.to_excel(data_path_save, index=False)

selected_df_sec.groupby('IDX')['CODE'].value_counts()


## 9. Correlation and reduced-feature sensitivity

Calculate Pearson correlations among the 25 discovery features. The saved lower-triangle plot annotates correlations with |r| ≥ 0.60.

In [ ]:
features = ['Anhedonie_t0', 'Anhedonie_t1', 'Anhedonie_t2', 'Anhedonie_t3',
       'Anhedonie_t4', 'Depression_t0', 'Depression_t1', 'Depression_t2',
       'Depression_t3', 'Depression_t4', 'Anxiety_t0', 'Anxiety_t1',
       'Anxiety_t2', 'Anxiety_t3', 'Anxiety_t4', 'Mood T0', 'Mood T1',
       'Mood T2', 'Mood T3', 'Mood T4', 'Stress T0', 'Stress T1', 'Stress T2',
       'Stress T3', 'Stress T4']

corr_matrix = selected_df[features].corr(method="pearson")

plt.figure(figsize=(7, 7))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True
)

plt.title("Correlation Matrix of Clustering Features")
plt.tight_layout()
plt.show()


In [ ]:
def set_global_plot_style():

    sns.set_context("paper", font_scale=1)

    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=10)
    plt.rc('axes', titlesize=10)
    plt.rc('axes', labelsize=10)
    plt.rc('xtick', labelsize=10)
    plt.rc('ytick', labelsize=10)
    plt.rc('legend', fontsize=10)
    plt.rc('figure', titlesize=10)

set_global_plot_style()

features = [
    'Anhedonie_t0', 'Anhedonie_t1', 'Anhedonie_t2', 'Anhedonie_t3', 'Anhedonie_t4',
    'Depression_t0', 'Depression_t1', 'Depression_t2', 'Depression_t3', 'Depression_t4',
    'Anxiety_t0', 'Anxiety_t1', 'Anxiety_t2', 'Anxiety_t3', 'Anxiety_t4',
    'Mood T0', 'Mood T1', 'Mood T2', 'Mood T3', 'Mood T4',
    'Stress T0', 'Stress T1', 'Stress T2', 'Stress T3', 'Stress T4'
]

corr_matrix = selected_df[features].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 10))

# Hide duplicate correlations in the upper triangle.
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool),
    k=1
)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={
        "label": "Pearson's r",
        "shrink": 0.75
    },
    ax=ax
)

for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):

        r = corr_matrix.iloc[i, j]

        if j < i and abs(r) >= 0.60:
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{r:.2f}",
                ha="center",
                va="center",
            )

short_labels = [
    'Anhedonia T0', 'Anhedonia T1', 'Anhedonia T2', 'Anhedonia T3', 'Anhedonia T4',
    'Depression T0', 'Depression T1', 'Depression T2', 'Depression T3', 'Depression T4',
    'Anxiety T0', 'Anxiety T1', 'Anxiety T2', 'Anxiety T3', 'Anxiety T4',
    'Mood T0', 'Mood T1', 'Mood T2', 'Mood T3', 'Mood T4',
    'Stress T0', 'Stress T1', 'Stress T2', 'Stress T3', 'Stress T4'
]

ax.set_xticklabels(
    short_labels,
    rotation=90,
)

ax.set_yticklabels(
    short_labels,
    rotation=0,
)

ax.set_xlabel("")
ax.set_ylabel("")

cbar = ax.collections[0].colorbar
cbar.ax.tick_params()
cbar.set_label("Pearson's r")

sns.despine(left=True, bottom=True)

plt.tight_layout()

output_path = OUTPUT_PLOTS / 'Supplementary_Figure_Correlation_Matrix.png'
plt.savefig(
    output_path,
    dpi=600,
    bbox_inches="tight",
    format="png"
)

plt.show()


Remove Mood T3 and Stress T3, refit a two-cluster solution with 23 features, and compare it with the original solution using ARI. This fit uses the current random state; it does not reset a seed locally. Diagnosis percentages are calculated within diagnosis.

In [ ]:
original_IDX = selected_df["IDX"].copy().values

print("Original cluster sizes:")
print(pd.Series(original_IDX).value_counts().sort_index())

features_to_remove = ["Mood T3", "Stress T3"]

selected_df_reduced = selected_df.drop(
    columns=features_to_remove
).copy()

print("\nRemoved features:")
print(features_to_remove)

print("\nNumber of features after removal:")
print(len(selected_df_reduced.columns))

feature_columns = [
    col for col in selected_df_reduced.columns
    if col not in ["IDX", "CODE", "ID"]
]

data_reduced = selected_df_reduced[feature_columns].values.astype(np.float32)
print(selected_df_reduced[feature_columns].columns)
print("\nReduced data shape:")
print(data_reduced.shape)

centroids_reduced, _ = kmeans(data_reduced, 2)
IDX_reduced, _ = vq(data_reduced, centroids_reduced)

print("\nReduced-feature cluster sizes:")
print(pd.Series(IDX_reduced).value_counts().sort_index())

selected_df_reduced["IDX_reduced"] = IDX_reduced

ARI_reduced = adjusted_rand_score(
    original_IDX,
    IDX_reduced
)

cluster_code_counts = (
    selected_df_reduced
    .groupby(["IDX_reduced", "CODE"])
    .size()
    .unstack(fill_value=0)
)

print("\nDiagnosis counts per reduced-feature cluster:")
print(cluster_code_counts)

cluster_code_percentages = (
    cluster_code_counts
    .div(cluster_code_counts.sum(axis=0), axis=1)
    * 100
)

print("\nDiagnosis percentages per reduced-feature cluster:")
print(cluster_code_percentages.round(1))
print("\nAdjusted Rand Index:")
print(f"ARI = {ARI_reduced:.4f}")


In [ ]:
selected_df_reduced["CODE"] = selected_df["CODE"].values
selected_df_reduced["ID"] = selected_df["ID"].values

custom_palette = {
    'PPD': '#FF7F50',
    'ND':  '#0056A0',
    'AS':  '#2C9F6D'
}

hue_order = ['ND', 'AS', 'PPD']

diagnosis_totals = (
    selected_df_reduced['CODE']
    .value_counts()
    .to_dict()
)

plt.figure(figsize=(8, 6))

ax = sns.countplot(
    data=selected_df_reduced,
    x='IDX_reduced',
    hue='CODE',
    palette=custom_palette,
    hue_order=hue_order
)

ax.set_xticklabels(['Cluster 1', 'Cluster 2'])

for i, container in enumerate(ax.containers):

    diag = hue_order[i]
    total = diagnosis_totals.get(diag, 1)

    for bar in container:

        height = bar.get_height()

        if height > 0:
            percentage = (height / total) * 100

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 1,
                f'{percentage:.1f}%',
                ha='center',
                va='bottom',
                fontsize=12
            )

ax.set_xlabel("")
ax.set_ylabel(
    "Percentage within diagnosis",
    fontsize=13
)

ax.set_yticks([])

sns.despine()

ax.legend(
    title='Diagnosis',
    fontsize=11,
    title_fontsize=12
)

plt.title(
    'Diagnosis distribution per Cluster - Discovery Cohort\n'
    'After removing Mood T3 and Stress T3',
    fontsize=16
)

plt.tight_layout()
plt.show()


### Bootstrap the reduced-feature solution

With seed 42, resample the actual number of participants 10,000 times and compare each refitted solution against the reduced-feature reference labels using ARI. Report the mean, median, and 2.5th–97.5th percentile range.

In [ ]:
np.random.seed(42)

n = data_reduced.shape[0]

ari_bootstrap = []

for i in range(10000):

    # Resample participants with replacement.
    idx = np.random.choice(n, n, replace=True)

    boot_data = data_reduced[idx]

    boot_centroids, _ = kmeans(boot_data, 2)
    boot_labels, _ = vq(boot_data, boot_centroids)

    ari_bootstrap.append(
        adjusted_rand_score(
            IDX_reduced[idx],
            boot_labels
        )
    )

print("\nBootstrap stability of 23-feature solution:")
print("Mean ARI:", np.mean(ari_bootstrap))
print("Median ARI:", np.median(ari_bootstrap))
print(
    "95% CI:",
    np.percentile(ari_bootstrap, [2.5, 97.5])
)
